# v4 Notebook B - experts (Llama-Guard-3-8B, Granite-Guardian-4.1-8B), two views
Run AFTER A (needs `v4_data.json`, `v4_pool.json` in OUT; upload A's output or run in the same persistent dir). T4 x2 if available. Output: `v4_B_experts.npz`.

In [ ]:
# Cell 1 - setup (Kaggle T4; Internet ON; Secrets: HF_TOKEN, AGENT_API_KEY)
import os, sys, subprocess, glob
KAGGLE = os.path.isdir('/kaggle/working')
DEST = ('/kaggle/working' if KAGGLE else '/content') + '/vyuha_src'
if not os.path.isdir(f'{DEST}/.git'):
    subprocess.run(['git','clone','--depth','1','https://github.com/g25ait2149/vyuha.git',DEST], check=False)
else:
    subprocess.run(['git','-C',DEST,'pull','--ff-only'], check=False)
hits = glob.glob(DEST+'/**/vyuha/__init__.py', recursive=True)
assert hits, 'clone failed - turn Internet ON'
ROOT = os.path.dirname(os.path.dirname(hits[0])); sys.path.insert(0, ROOT); os.chdir(ROOT)
subprocess.run('pip -q install -U "bitsandbytes>=0.46.1" accelerate "transformers>=4.44" datasets scikit-learn sentencepiece "protobuf<5" huggingface_hub', shell=True)
tok = ''
try:
    if KAGGLE:
        from kaggle_secrets import UserSecretsClient; tok = UserSecretsClient().get_secret('HF_TOKEN')
    else:
        from google.colab import userdata; tok = userdata.get('HF_TOKEN')
except Exception as e:
    print('HF_TOKEN missing:', repr(e)[:100])
os.environ['HF_TOKEN'] = os.environ['HUGGINGFACE_HUB_TOKEN'] = tok or ''
from huggingface_hub import login
if tok: login(tok)
import torch; print('GPUs:', torch.cuda.device_count(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
OUT = ('/kaggle/working' if KAGGLE else '/content') + '/v4'; os.makedirs(OUT, exist_ok=True)
print('OUT =', OUT)

In [ ]:
# Cell 2 - two experts x two views, one expert per GPU (T4 x2). Reads the pool texts saved by notebook A.
import json, os, numpy as np, torch
from vyuha.normalize.normalize import normalize
from vyuha.cascade.guards import load_model, forced_prefix, bucketed_scores
from vyuha.cascade.data import ensure_files
ensure_files(OUT)                                   # self-provision A's data/pool if not already present
EVAL = json.load(open(f'{OUT}/v4_data.json'))['eval_seeds']
WC_CALIB = json.load(open(f'{OUT}/v4_data.json'))['wc_calib']
WC_TEST  = json.load(open(f'{OUT}/v4_data.json'))['wc_test']
HARD     = json.load(open(f'{OUT}/v4_data.json'))['hard']
POOL = json.load(open(f'{OUT}/v4_pool.json'))['texts']; NS, NC = len(POOL), 60
flat = [t for row in POOL for t in row]
V = lambda xs: [normalize(t, full=True, version=2) for t in xs]
probes = [t for t in EVAL+WC_CALIB if len(t)<300][:5]
OUTF = f'{OUT}/v4_B_experts.npz'; store = dict(np.load(OUTF)) if os.path.exists(OUTF) else {}
def put(k,v): store[k]=np.asarray(v); np.savez(OUTF, **store)

def score_expert(mid, kind, tag, device):
    if f'{tag}/l0/pool' in store: print('[resume]', tag); return
    tok, m = load_model(mid, True, device=device)
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()): pre = forced_prefix(tok, m, kind, probes)
    prog = lambda k,n: print(f'  {tag} {k}/{n}', end='\r', flush=True)
    for view, texts in [('l0', None), ('raw', None)]:
        def enc_view(xs): return V(xs) if view=='l0' else list(xs)
        put(f'{tag}/{view}/ben_wc_calib', bucketed_scores(tok,m, enc_view(WC_CALIB), kind, pre, progress=prog))
        put(f'{tag}/{view}/ben_wc_test',  bucketed_scores(tok,m, enc_view(WC_TEST),  kind, pre, progress=prog))
        put(f'{tag}/{view}/ben_hard',     bucketed_scores(tok,m, enc_view(HARD),     kind, pre, progress=prog))
        put(f'{tag}/{view}/orig',         bucketed_scores(tok,m, enc_view(EVAL),     kind, pre, progress=prog))
        put(f'{tag}/{view}/pool',         bucketed_scores(tok,m, enc_view(flat),     kind, pre, progress=prog).reshape(NS,NC))
        if tag=='E_GR': break      # Granite: L0 view only (secondary); raw view is E_LG only (prereg S1.3)
    m=tok=None; import gc; gc.collect(); torch.cuda.empty_cache(); print('  done', tag, flush=True)

# E_LG on GPU0, E_GR on GPU1 (sequential here; set device per the prereg - both fit a single T4 at 4-bit if only one GPU)
ndev = torch.cuda.device_count()
score_expert('meta-llama/Llama-Guard-3-8B', 'llamaguard', 'E_LG', 0)
score_expert('ibm-granite/granite-guardian-4.1-8b', 'granite41', 'E_GR', 1 if ndev>1 else 0)
print('experts scored:', sorted({k.split("/")[0] for k in store}))